In [ ]:
!pip install transformers
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 5.9 MB/s eta 0:00:00


In [ ]:
import os
import requests
from PIL import Image
from transformers import pipeline
import pandas as pd

In [ ]:
from google.colab import drive
import os
import gdown
drive.mount('/content/drive', force_remount=True)

TRAFFIC_MONITORING_DIR = '/content/drive/My Drive/Traffic_monitoring'

# Create the directory if it does not exist
if not os.path.exists(TRAFFIC_MONITORING_DIR):
    os.makedirs(TRAFFIC_MONITORING_DIR)
    print(f"Directory '{TRAFFIC_MONITORING_DIR}' created.")
else:
    print(f"Directory '{TRAFFIC_MONITORING_DIR}' already exists.")


traffic_image_link = 'https://drive.google.com/uc?id=1wzFdCRskjuGN8QpmZ1-v8juwgZ-rlx-H'

# Specify the destination path where you want to save the file
TRAFFIC_IMAGE_INPUT = TRAFFIC_MONITORING_DIR + '/Wed1-20-20.csv'

# Download the file
gdown.download(traffic_image_link, TRAFFIC_IMAGE_INPUT, quiet=False)

Mounted at /content/drive
Directory '/content/drive/My Drive/Traffic_monitoring' already exists.


Downloading...
From: https://drive.google.com/uc?id=1wzFdCRskjuGN8QpmZ1-v8juwgZ-rlx-H
To: /content/drive/My Drive/Traffic_monitoring/Wed1-20-20.csv
100%|██████████| 2.94k/2.94k [00:00<00:00, 7.43MB/s]


'/content/drive/My Drive/Traffic_monitoring/Wed1-20-20.csv'

In [ ]:
#Import a previous generated and uploaded csv from dropbox
#Remember to set 1 instead of 0 in the last part of the URL
def read_lta_csv(google_url):
  df = pd.read_csv(google_url, sep=',', lineterminator='\r')
  #print(df)
  images = df.loc[:,'Image']
  cam_id = df.loc[:,'Cam_id']
  timestamp_sg_time = df.loc[:,'Timestamp_sg_time']
  url = df.loc[:,'Image']

  return images,cam_id,timestamp_sg_time,url


In [ ]:
#https://huggingface.co/facebook/detr-resnet-50
from transformers import DetrFeatureExtractor, DetrForObjectDetection
import torch
from PIL import Image
import requests

feature_extractor = DetrFeatureExtractor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

DETECTION_THRESHOLD = 0.2
NEEDED_LABELS = ["bus", "car", "truck"]
DETECTION_OBJ = {"bus": 6, "car": 3, "truck":8 }

def detect_object(image):
    inputs = feature_extractor(images=image, return_tensors="pt")
    outputs = model(**inputs)

    # convert outputs (bounding boxes and class logits) to COCO API
    target_sizes = torch.tensor([image.size[::-1]])
    results = feature_extractor.post_process_object_detection(outputs, target_sizes=target_sizes)[0]
    return results


def process_detected_obj(image_area, results):
    detection_classes = []
    detection_label = []
    detection_scores = []
    xmin_list = []
    ymin_list = []
    xmax_list = []
    ymax_list = []
    bb = []
    proportion_bb = []

    for score, label, boxlist in zip(results["scores"], results["labels"], results["boxes"]):

        # let's only keep detections with score > DETECTION_THRESHOLD
        label = model.config.id2label[label.item()]

        if (score.item() > DETECTION_THRESHOLD and label in NEEDED_LABELS):
            box = [int(j) for j in boxlist.tolist()]

            # Get actual box
            ###https://github.com/huggingface/transformers/blob/main/src/transformers/pipelines/object_detection.py
            xmin, ymin, xmax, ymax =  box[0], box[1], box[2], box[3]
            bounding_box_area = (xmax - xmin) * (ymax - ymin)
            proportion_bounding_box = (bounding_box_area / image_area)*100

            detection_classes.append(DETECTION_OBJ[label])
            detection_scores.append(score.item())
            xmin_list.append(xmin)
            ymin_list.append(ymin)
            xmax_list.append(xmax)
            ymax_list.append(ymax)

            bb.append(bounding_box_area)
            proportion_bb.append(proportion_bounding_box)


    return detection_classes,detection_scores, ymin_list,xmin_list,ymax_list,xmax_list, bb, proportion_bb


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/models/detr/feature_extraction_detr.py:38: FutureWarning: The class DetrFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use DetrImageProcessor instead.
  warnings.warn(


config.json:   0%|          | 0.00/4.59k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
import itertools

def process_data(images,cam_id,timestamp_sg_time,url):
    camera_number= str(cam_id)
    df_app = []
    for i in range(len(images)):
        camera = str(cam_id)
        try:
            url =  images[i]
            image = Image.open(requests.get(url, stream=True).raw)
            width, height = image.size
            image_area = width * height

            results = detect_object(image)
            detection_classes, scores, ymin, xmin, ymax, xmax,bb_area, proportion_bb= process_detected_obj(image_area,results)

            df = pd.DataFrame(list(zip(itertools.repeat(i),
                            itertools.repeat(camera_number),
                            detection_classes,
                            scores,
                            itertools.repeat(timestamp_sg_time[i]),
                            itertools.repeat(url),
                            ymin,xmin, ymax, xmax,
                            bb_area, proportion_bb)),
            columns =['Image_id', 'Cam_ID','Detection_Class',
                    'Detection_Score','Timestamp_sg_time','URL',
                    'ymin', 'xmin', 'ymax', 'xmax','Bounding_box_area','Proportion_bounding_box_0_100'])
            #print(df)
            df_app.append(df)
        except Exception as e:
          #print('break time: ', times[i])
            print("Exception :: ",str(e))
            print("Exception Date :: ",timestamp_sg_time[i])
            print('break time:', i)

    df_simple = pd.concat(df_app)
    a = df_simple.dtypes
    #print (a)
    print(df_simple.head())

    return df_simple, camera_number


In [ ]:
def store_in_file(df_simple, camera_number):

    #To save contents to google drive
    file_name = "output_huggingface_transformer.csv"
    output_file_full_path = os.path.join(TRAFFIC_MONITORING_DIR, file_name)
    df_simple.to_csv(output_file_full_path)

In [ ]:
images,cam_id,timestamp_sg_time,url = read_lta_csv(TRAFFIC_IMAGE_INPUT)
df_simple,  camera_number = process_data(images,cam_id,timestamp_sg_time,url)
store_in_file(df_simple, camera_number)

Exception ::  Invalid URL 'nan': No scheme supplied. Perhaps you meant https://nan?
Exception Date ::  nan
break time: 24
  Image_id                                             Cam_ID Detection_Class  \
0        0  0     \n1701\n1     \n1701\n2     \n1701\n3   ...               3   
1        0  0     \n1701\n1     \n1701\n2     \n1701\n3   ...               3   
2        0  0     \n1701\n1     \n1701\n2     \n1701\n3   ...               3   
3        0  0     \n1701\n1     \n1701\n2     \n1701\n3   ...               8   
4        0  0     \n1701\n1     \n1701\n2     \n1701\n3   ...               3   

   Detection_Score    Timestamp_sg_time  \
0         0.939477  2020-01-20 00:00:00   
1         0.996119  2020-01-20 00:00:00   
2         0.500721  2020-01-20 00:00:00   
3         0.687339  2020-01-20 00:00:00   
4         0.974669  2020-01-20 00:00:00   

                                                 URL ymin xmin ymax xmax  \
0  https://images.data.gov.sg/api/traffic-images/...  45